In [ ]:
#Final code all features
import os
import re
import math
import pandas as pd
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning, XMLParsedAsHTMLWarning
from collections import Counter
from urllib.parse import urlparse
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# === Helper Functions ===
def calculate_entropy(text):
    if not text:
        return 0
    probabilities = [n / len(text) for n in Counter(text).values()]
    return -sum(p * math.log2(p) for p in probabilities)

def max_tag_depth(tag):
    if not hasattr(tag, 'contents') or not tag.contents:
        return 1
    return 1 + max((max_tag_depth(child) for child in tag.contents if hasattr(child, 'name')), default=0)

def extract_additional_url_features(all_links):
    valid_links = [link for link in all_links if link]
    digit_counts = sum(sum(c.isdigit() for c in link) for link in valid_links)
    punct_counts = sum(sum(c in "/.-_=?;&" for c in link) for link in valid_links)
    subdomain_counts = sum(urlparse(link).hostname.count('.') if urlparse(link).hostname else 0 for link in valid_links)
    url_lengths = [len(link) for link in valid_links]
    hostname_digit_ratios = [
        sum(c.isdigit() for c in urlparse(link).hostname) / len(urlparse(link).hostname)
        if urlparse(link).hostname else 0 for link in valid_links
    ]

    return {
        "url_digit_count": digit_counts,
        "url_punct_char_count": punct_counts,
        "url_avg_length": sum(url_lengths) / len(url_lengths) if url_lengths else 0,
        "url_avg_subdomain_count": subdomain_counts / len(valid_links) if valid_links else 0,
        "hostname_digit_ratio_avg": sum(hostname_digit_ratios) / len(hostname_digit_ratios) if hostname_digit_ratios else 0,
        "min_link_length": min(url_lengths) if url_lengths else 0,
        "max_link_length": max(url_lengths) if url_lengths else 0
    }

# === HTML Feature Extractor ===
def extract_features_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text()
    script_text = "\n".join(s.get_text() for s in soup.find_all("script"))
    all_links = [a.get("href") or a.get("src") for a in soup.find_all(["a", "link", "script", "iframe", "img"]) if a.get("href") or a.get("src")]
    words = re.findall(r'\w+', text)
    total_words = len(words)

    redirect_patterns = [
        r'<meta\s+http-equiv=["\']refresh["\']',
        r'window\.location',
        r'document\.location',
        r'location\.replace',
        r'location\.assign',
        r'top\.location'
    ]

    hidden_iframes = [iframe for iframe in soup.find_all("iframe")
                      if iframe.get("style", "").find("display:none") != -1 or
                      iframe.get("height") == "0" or iframe.get("width") == "0"]

    suspicious_keywords = ["login", "password", "verify", "account", "secure", "bank"]
    suspicious_count = sum(text.lower().count(word) for word in suspicious_keywords)

    features = {
        "file_size": len(html),
        "line_count": html.count("\n"),
        "entropy": calculate_entropy(html),
        "script_entropy": calculate_entropy(script_text),
        "tag_count": len(soup.find_all()),
        "unique_tag_count": len(set(tag.name for tag in soup.find_all())),
        "script_count": len(soup.find_all("script")),
        "form_count": len(soup.find_all("form")),
        "iframe_count": len(soup.find_all("iframe")),
        "hidden_iframe_count": len(hidden_iframes),
        "external_links_count": sum(1 for link in all_links if urlparse(link).netloc),
        "mailto_link_count": sum(1 for link in all_links if link and link.startswith("mailto:")),
        "base64_string_count": len(re.findall(r"(?:[A-Za-z0-9+/]{4}){10,}", html)),
        "html_comment_count": html.count("<!--"),
        "max_tag_nesting_depth": max_tag_depth(soup),
        "eval_in_script_blocks": script_text.lower().count("eval("),
        "function_count": len(re.findall(r"\bfunction\b", script_text, re.IGNORECASE)),
        "total_script_characters": len(script_text),
        "suspicious_word_count": suspicious_count,
        "keywords_to_words_ratio": suspicious_count / total_words if total_words else 0,
        "whitespace_ratio": text.count(" ") / len(text) if text else 0,
        "escaped_char_count": len(re.findall(r"(\\x[0-9a-fA-F]{2}|&#x[0-9a-fA-F]+;)", html)),
        "hex_encoding_rate": len(re.findall(r"(\\x[0-9a-fA-F]{2}|&#x[0-9a-fA-F]+;)", html)) / len(html) if html else 0,
        "noscript_count": len(soup.find_all("noscript")),
        "embedded_js_count": sum(1 for s in soup.find_all("script") if not s.get("src")),
        "external_js_count": sum(1 for s in soup.find_all("script") if s.get("src")),
        "internal_link_count": sum(1 for link in all_links if link and not urlparse(link).netloc),
        "external_link_count": sum(1 for link in all_links if link and urlparse(link).netloc),
        "img_count": len(soup.find_all("img")),
        "html_whitespace_ratio": html.count(" ") / len(html) if html else 0,
        "redirect_mechanism_count": sum(len(re.findall(pat, html, re.IGNORECASE)) for pat in redirect_patterns),
        "event_attachment_count": len(re.findall(r'on\w+=', html, re.IGNORECASE)),
        "object_tag_count": len(soup.find_all("object"))
    }

    features.update(extract_additional_url_features(all_links))
    return features

# === File Reader ===
def extract_features_from_bin_file(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw_html = f.read()

    if len(raw_html.strip()) < 100:
        raise ValueError("File too small or empty")

    try:
        soup = BeautifulSoup(raw_html, "html.parser")
    except Exception:
        soup = BeautifulSoup(raw_html, "xml")

    features = extract_features_from_html(str(soup))
    features["file_path"] = filepath
    return features

# === Incremental Batch Processor ===
def process_bin_folder_for_features(folder_path, output_csv_path):
    total_files = 0
    failed_files = 0
    processed_files = set()

    # Load previously processed file paths
    if os.path.exists(output_csv_path):
        try:
            existing_df = pd.read_csv(output_csv_path, usecols=["file_path"])
            processed_files = set(existing_df["file_path"].tolist())
            print(f"🔁 Resuming from previous run. Already processed {len(processed_files)} files.")
        except Exception as e:
            print(f"⚠️ Failed to read existing CSV: {e}")

    first_write = not os.path.exists(output_csv_path)

    with open(output_csv_path, 'a', encoding='utf-8', newline='') as f:
        for root, _, files in os.walk(folder_path):
            for fname in files:
                if fname.endswith((".html", ".htm", ".bin")):
                    fpath = os.path.join(root, fname)
                    if fpath in processed_files:
                        continue

                    total_files += 1
                    try:
                        feats = extract_features_from_bin_file(fpath)
                        df_row = pd.DataFrame([feats])
                        df_row.to_csv(f, index=False, header=first_write)
                        first_write = False
                        print(f"✅ Processed: {fname}")
                    except Exception as e:
                        failed_files += 1
                        print(f"❌ Skipped {fpath}: {e}")

    print(f"\n✅ Extracted features from {total_files - failed_files} out of {total_files} files (skipped {failed_files})")
    print(f"📁 Output saved to: {output_csv_path}")

# === RUN THIS ===
input_path = "Put your input path here"
output_path = "Put your output path here"


process_bin_folder_for_features(input_path, output_path)